# Smart Shelf CV — เทียบรูป Reference (ถูกต้อง) กับรูปจริง

แนวทาง: ไม่ต้องพิมพ์ planogram เป็น JSON เอง — ถ่ายรูปชั้นวางตอนวางถูกต้อง 1 รูป (**reference**) แล้วเอาโมเดลตัวเดียวกัน detect ทั้ง reference กับรูปจริงที่จะเช็ค (**test**) จากนั้นเทียบผล detection สองชุดนี้กันโดยตรง

ก่อนรัน ต้องมี:
- API key ของ Roboflow (โมเดล SKU-110k ที่ทดสอบไว้)
- รูป reference (ชั้นวางตอนถูกต้อง) 1 รูป
- รูป test (ชั้นวางที่จะเช็ค — เช่นเคสที่จงใจเอาของออก/สลับที่) 1 รูป
- **สำคัญ:** ถ่าย 2 รูปนี้จากมุมกล้อง/ระยะเดียวกันให้มากที่สุด ไม่งั้นตำแหน่ง grid จะเทียบกันไม่ตรง

## Step 1: ติดตั้ง library

In [ ]:
!pip install requests pillow numpy matplotlib -q

## Step 2: ตั้งค่า

แก้ค่าตรงนี้ให้ตรงกับของจริงก่อนรัน

In [ ]:
API_KEY = 'ใส่ API key ตรงนี้'
MODEL_ENDPOINT = 'sku-110k/2'
CONFIDENCE_THRESHOLD = 40

REFERENCE_IMAGE_PATH = 'shelf_reference.jpg'
TEST_IMAGE_PATH = 'shelf_test.jpg'

GRID_ROWS = 2
GRID_COLS = 4

## Step 3: ฟังก์ชันหลัก (เรียก detection + เช็คสี + แปลงเป็น grid)

เขียนเป็นฟังก์ชันเดียวเพื่อเรียกซ้ำได้ทั้งกับรูป reference และรูป test

In [ ]:
import requests
import base64
import numpy as np
from PIL import Image

REFERENCE_COLORS = {
    'red': (200, 30, 30),
    'orange': (230, 130, 30),
    'yellow': (220, 200, 40),
    'brown': (100, 60, 30),
    'clear': (210, 220, 225),
    'green': (40, 140, 60),
    'blue': (30, 90, 180),
}

def call_detection_api(image_path):
    with open(image_path, 'rb') as f:
        image_data = base64.b64encode(f.read()).decode('utf-8')
    url = f'https://detect.roboflow.com/{MODEL_ENDPOINT}'
    params = {'api_key': API_KEY, 'confidence': CONFIDENCE_THRESHOLD}
    response = requests.post(url, params=params, data=image_data,
                              headers={'Content-Type': 'application/x-www-form-urlencoded'})
    response.raise_for_status()
    return response.json()['predictions']

def get_dominant_color(crop):
    pixels = np.array(crop).reshape(-1, 3).astype(float)
    brightness = pixels.mean(axis=1)
    mask = (brightness > 20) & (brightness < 235)
    filtered = pixels[mask] if mask.sum() > 10 else pixels
    return tuple(filtered.mean(axis=0))

def classify_color(rgb):
    best_name, best_dist = None, float('inf')
    for name, ref in REFERENCE_COLORS.items():
        dist = sum((a - b) ** 2 for a, b in zip(rgb, ref))
        if dist < best_dist:
            best_name, best_dist = name, dist
    return best_name

def analyze_image(image_path):
    detections = call_detection_api(image_path)
    image = Image.open(image_path).convert('RGB')
    img_w, img_h = image.size

    items = []
    for det in detections:
        cx, cy, w, h = det['x'], det['y'], det['width'], det['height']
        left, top = cx - w / 2, cy - h / 2
        right, bottom = cx + w / 2, cy + h / 2
        crop = image.crop((left, top, right, bottom))

        dominant_rgb = get_dominant_color(crop)
        row = min(int(cy / img_h * GRID_ROWS), GRID_ROWS - 1)
        col = min(int(cx / img_w * GRID_COLS), GRID_COLS - 1)

        items.append({
            'center_x': cx, 'center_y': cy, 'width': w, 'height': h,
            'row': row, 'col': col,
            'color': classify_color(dominant_rgb),
            'confidence': det['confidence'],
        })
    return items, image, img_w, img_h

## Step 4: รัน detection กับรูป reference และรูป test

In [ ]:
reference_items, reference_image, ref_w, ref_h = analyze_image(REFERENCE_IMAGE_PATH)
print(f'Reference: เจอ {len(reference_items)} ชิ้น')

test_items, test_image, test_w, test_h = analyze_image(TEST_IMAGE_PATH)
print(f'Test: เจอ {len(test_items)} ชิ้น')

## Step 5: เทียบผล reference กับ test ทีละ grid cell

- **correct** — สีตรงกันทั้งสองรูป
- **misplaced** — cell นี้มีของทั้งคู่ แต่สีไม่ตรงกัน (วางสินค้าผิดชนิด)
- **missing** — reference มีของ แต่ test cell นี้ว่าง (ของหาย/ยังไม่ได้เติม)
- **extra** — test มีของ แต่ reference cell นี้ไม่มี (วางของแปลกปลอมเพิ่ม)

In [ ]:
def to_grid_map(items):
    grid = {}
    for it in items:
        grid[(it['row'], it['col'])] = it
    return grid

reference_grid = to_grid_map(reference_items)
test_grid = to_grid_map(test_items)

all_cells = set(reference_grid.keys()) | set(test_grid.keys())

report = []
for row, col in sorted(all_cells):
    ref_item = reference_grid.get((row, col))
    test_item = test_grid.get((row, col))

    if ref_item and test_item:
        status = 'correct' if ref_item['color'] == test_item['color'] else 'misplaced'
    elif ref_item and not test_item:
        status = 'missing'
    else:
        status = 'extra'

    report.append({
        'row': row, 'col': col,
        'reference_color': ref_item['color'] if ref_item else None,
        'test_color': test_item['color'] if test_item else None,
        'status': status,
    })

for item in report:
    print(item)

## Step 6: วาดผลลัพธ์ทับรูป test (เขียว = ถูก, แดง = ผิด/ขาด/แปลกปลอม)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(test_image)

for det in test_items:
    left = det['center_x'] - det['width'] / 2
    top = det['center_y'] - det['height'] / 2
    rect = patches.Rectangle((left, top), det['width'], det['height'],
                              linewidth=2, edgecolor='cyan', facecolor='none')
    ax.add_patch(rect)
    ax.text(left, top - 5, det['color'], color='cyan', fontsize=9)

cell_w, cell_h = test_w / GRID_COLS, test_h / GRID_ROWS
for item in report:
    color = 'lime' if item['status'] == 'correct' else 'red'
    x, y = item['col'] * cell_w, item['row'] * cell_h
    rect = patches.Rectangle((x, y), cell_w, cell_h,
                              linewidth=2, edgecolor=color, facecolor='none', linestyle='--')
    ax.add_patch(rect)
    ax.text(x + 5, y + 20, item['status'], color=color, fontsize=10, weight='bold')

plt.axis('off')
plt.show()